In [40]:
import requests
import re
import pandas as pd
import numpy as np
API_KEY = 'f91688b2469810e18dbf6649b7d462fe'
sport_key = 'soccer_epl'
region = 'uk,eu'
url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds/?apiKey={API_KEY}&regions={region}"
output_annotated = "../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv"
df.to_csv(output_annotated, index=False)


pattern = re.compile(r'[A-Z0-9]+[HAD]$|[A-Z0-9]+(?:>|<).*|ODDS', re.IGNORECASE)
odds_cols = [col for col in df.columns if pattern.match(col)]
print("Odds columns in df:")
for col in odds_cols:
    print(col)

    
response = requests.get(url)
data = response.json()
#print("Requests remaining:", response.headers.get("x-requests-remaining"))
#print("Requests used:", response.headers.get("x-requests-used"))
#print("Requests cost for this call:", response.headers.get("x-requests-last"))
#for event in data:
#    print(f"\n{event['home_team']} vs {event['away_team']}:")
#    for bookmaker in event.get('bookmakers', []):
#        print(f"  Bookmaker: {bookmaker['key']} ({bookmaker.get('title', '')})")
#        print("    Markets:", [market['key'] for market in bookmaker.get('markets', [])])

Odds columns in df:
Requests remaining: 475
Requests used: 25
Requests cost for this call: 2


In [45]:
import requests
import re
import pandas as pd
import numpy as np

# 1. Load the annotated DataFrame
output_annotated = "../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv"
df = pd.read_csv(output_annotated)

# 2. Identify and print odds columns by pattern (whether filled or not)
pattern = re.compile(r'[A-Z0-9]+[HAD]$|[A-Z0-9]+(?:>|<).*|ODDS', re.IGNORECASE)
odds_cols = [col for col in df.columns if pattern.match(col)]
print("Odds columns in df:")
for col in odds_cols:
    print(col)

# 3. Define how to fill each odds column (expand as needed)
odds_col_map = {
    "B365H":      ("bet365", "h2h", "home"),
    "B365D":      ("bet365", "h2h", "draw"),
    "B365A":      ("bet365", "h2h", "away"),
    "B365>2.5":   ("bet365", "totals", "Over", 2.5),
    "B365<2.5":   ("bet365", "totals", "Under", 2.5),
    # Add more as needed, e.g. "PSH": ("pinnacle", "h2h", "home"), etc.
}

# Ensure those columns exist (initialize if not)
for col in odds_col_map:
    if col not in df.columns:
        df[col] = np.nan

# 4. Download fresh odds data from API
API_KEY = 'f91688b2469810e18dbf6649b7d462fe'  # Your API Key
sport_key = 'soccer_epl'
region = 'uk,eu'
markets = 'h2h,totals'
url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds/?apiKey={API_KEY}&regions={region}&markets={markets}"
response = requests.get(url)
api_data = response.json()

# Uncomment to see your current quota at any time:
#print("Requests remaining:", response.headers.get("x-requests-remaining"))
#print("Requests used:", response.headers.get("x-requests-used"))
#print("Requests cost for this call:", response.headers.get("x-requests-last"))

# 5. Odds filling logic
FALLBACKS = {
    "bet365": ["bet365", "pinnacle", "williamhill"],
    "pinnacle": ["pinnacle", "bet365", "williamhill"],
    # Add more if needed for other bookmakers
}

API_TEAM_NORMALIZATION = lambda s: str(s).strip().lower()

def get_match_odds(event, bookmaker_key, market_key, outcome_name, point=None, ah_side=None):
    for bk in FALLBACKS.get(bookmaker_key, [bookmaker_key]):
        for bookmaker in event.get("bookmakers", []):
            if bookmaker["key"] == bk:
                for market in bookmaker.get("markets", []):
                    if market["key"] == market_key:
                        if market_key == "h2h":
                            for out in market["outcomes"]:
                                team = out["name"].strip().lower()
                                if outcome_name == "home" and team == event["home_team"].strip().lower():
                                    return out["price"]
                                elif outcome_name == "away" and team == event["away_team"].strip().lower():
                                    return out["price"]
                                elif outcome_name == "draw" and "draw" in team:
                                    return out["price"]
                        elif market_key == "totals" and point is not None:
                            for out in market["outcomes"]:
                                if (out["name"].lower() == outcome_name.lower() and
                                    float(out.get("point", 0)) == float(point)):
                                    return out["price"]
                        elif market_key == "spreads":
                            if ah_side is not None:
                                for out in market["outcomes"]:
                                    if (out["name"].lower() == ah_side.lower()
                                        and float(out.get("point", 0)) == float(point)):
                                        return out["price"]
                            if outcome_name is None and len(market['outcomes']):
                                return market["outcomes"][0].get("point")
    return np.nan

# 6. Loop through your fixture rows and fill odds columns
for idx, row in df.iterrows():
    home = API_TEAM_NORMALIZATION(row.get("HomeTeam", ""))
    away = API_TEAM_NORMALIZATION(row.get("AwayTeam", ""))
    event_found = [e for e in api_data
                   if API_TEAM_NORMALIZATION(e["home_team"]) == home
                   and API_TEAM_NORMALIZATION(e["away_team"]) == away]
    if not event_found:
        continue
    event = event_found[0]
    for col, mapping in odds_col_map.items():
        if len(mapping) == 3:  # h2h columns
            df.at[idx, col] = get_match_odds(event, mapping[0], mapping[1], mapping[2])
        elif len(mapping) == 4:  # totals columns
            df.at[idx, col] = get_match_odds(event, mapping[0], mapping[1], mapping[2], point=mapping[3])
        # Expand for Asian Handicap, etc if needed

# 7. Save output
df.to_csv("filled_with_odds.csv", index=False)
print("Odds columns populated with API data (with fallback where needed).")

# -- Extra diagnostic code for future debugging --
# To print a summary of which columns were filled:
#print(df[list(odds_col_map.keys())].head(10))
# To print odds per event from API:
#for event in api_data:
#    print(event['home_team'], "vs", event['away_team'])
#    for bookmaker in event.get('bookmakers', []):
#        print(' Bookmaker:', bookmaker['key'])
#        for market in bookmaker.get('markets', []):
#            print('  Market:', market['key'])
#            for outcome in market.get('outcomes', []):
#                print('   ', outcome)

Odds columns in df:
Month
1XBH
1XBD
1XBA
B365H
B365D
B365A
B365>2.5
B365<2.5
B365AHH
B365AHA
B365AH
BFH
BFD
BFA
BFEH
BFED
BFEA
BFDH
BFDD
BFDA
BMGMH
BMGMD
BMGMA
BVH
BVD
BVA
VCH
VCD
VCA
BSH
BSD
BSA
BWH
BWD
BWA
CLH
CLD
CLA
GBH
GBD
GBA
GB>2.5
GB<2.5
GBAHH
GBAHA
GBAH
IWH
IWD
IWA
LBH
LBD
LBA
LBAHH
LBAHA
LBAH
PSH
PSD
PSA
P>2.5
P<2.5
PAHH
PAHA
SOH
SOD
SOA
SBH
SBD
SBA
SJH
SJD
SJA
SYH
SYD
SYA
WHH
WHD
WHA
BbMxH
BbAvH
BbMxD
BbAvD
BbMxA
BbAvA
BbMx>2.5
BbAv>2.5
BbMx<2.5
BbAv<2.5
BbAH
BbAHh
BbMxAHH
BbAvAHH
BbMxAHA
BbAvAHA
MaxH
MaxD
MaxA
AvgH
AvgD
AvgA
Max>2.5
Max<2.5
Avg>2.5
Avg<2.5
MaxAHH
MaxAHA
AvgAHH
AvgAHA
AHh
HomeRecentGA
AwayRecentGA
Odds columns populated with API data (with fallback where needed).
